# OCR



---

## 1. Learning Objectives

- Explain the NLP processing pipeline.
- Implement tokenization, stopword removal, and TF-IDF.
- Train a text classifier on the 20 Newsgroups dataset.
- Generate a synthetic document image and extract text with EasyOCR.
- Understand PaddleOCR's scanned PDF workflow.
- Filter and analyze OCR confidence scores.

## 3. Concept / Theory

### NLP Pipeline
```
Raw Text → Tokenize → Lowercase → Remove Stopwords
    → Stemming/Lemmatization → Vectorize (TF-IDF)
    → Machine Learning Model → Output
```

### TF-IDF
- **TF (Term Frequency):** How often a word appears in a document
- **IDF (Inverse Document Frequency):** How rare the word is across all documents
- **TF-IDF = TF × IDF:** Common in document, rare overall → high score

### OCR (Optical Character Recognition)
Converts images of text into machine-readable text.

| OCR Engine | Strengths |
|---|---|
| **Tesseract** | Open-source, many languages |
| **EasyOCR** | Deep learning, 80+ languages, accurate |
| **PaddleOCR** | Very accurate, handles PDFs, multi-language |

### OCR Pipeline
```
Document Image → Grayscale → Denoise → Deskew
    → Text Detection → Text Recognition → Post-processing
```

In [ ]:
# Setup
!pip install -q scikit-learn easyocr pillow matplotlib pandas numpy opencv-python-headless

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import io, re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
print('Setup complete!')

## Practical Lab 1 – Text Processing: Tokenization & TF-IDF

### Objective
Process raw text into ML-ready numerical features.

### Explanation
Raw text must be converted to numbers before ML algorithms can use it. TF-IDF is the most widely used method for document-level text features.

In [ ]:
# Lab 1: Manual text processing
import re

def simple_tokenize(text):
    """Lowercase, remove punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = text.split()
    return tokens

stopwords = {'the','a','an','is','are','was','were','to','of','and','or','but','in','on','at','it','this','that','for','with','be','been','by','from','as'}

def remove_stopwords(tokens, stops):
    return [t for t in tokens if t not in stops]

# Test with sample documents
docs = [
    'Machine Learning is transforming every industry today',
    'Deep learning neural networks are powerful tools for AI applications',
    'Natural Language Processing enables computers to understand human text',
    'Computer vision systems can detect objects in images using deep learning',
]

print('Tokenization demo:')
for doc in docs[:2]:
    tokens = simple_tokenize(doc)
    filtered = remove_stopwords(tokens, stopwords)
    print(f'  Original: {doc[:50]}...')
    print(f'  Tokens:   {tokens}')
    print(f'  Filtered: {filtered}\n')

In [ ]:
# TF-IDF with sklearn
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=20,
    ngram_range=(1, 2)  # unigrams and bigrams
)

X_tfidf = vectorizer.fit_transform(docs)
print('TF-IDF matrix shape:', X_tfidf.shape)
print('\nVocabulary (top 20):', vectorizer.get_feature_names_out().tolist())

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
df_tfidf.index = [f'Doc{i+1}' for i in range(len(docs))]
print('\nTF-IDF Matrix:')
print(df_tfidf.round(3).to_string())

### Expected Output
```
TF-IDF matrix shape: (4, 20)
Each row = document; each column = word weight
```

## Practical Lab 2 – Text Classification with 20 Newsgroups

### Objective
Build a multi-class text classifier on a real news dataset.

### Explanation
The 20 Newsgroups dataset contains ~18,000 posts across 20 categories. We use TF-IDF + Logistic Regression — a strong baseline for text classification.

In [ ]:
# Lab 2: Load 20 Newsgroups (4 categories for speed)
categories = ['sci.space', 'sci.med', 'comp.graphics', 'rec.sport.hockey']
news = fetch_20newsgroups(subset='all', categories=categories,
                           remove=('headers','footers','quotes'))
print(f'Dataset: {len(news.data)} documents, {len(news.target_names)} categories')
print('Categories:', news.target_names)
print('\nSample document (first 300 chars):')
print(news.data[0][:300])

In [ ]:
# Train text classifier
X_tr, X_te, y_tr, y_te = train_test_split(
    news.data, news.target, test_size=0.2, random_state=42, stratify=news.target)

text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1,2), stop_words='english')),
    ('clf',   LogisticRegression(max_iter=500, C=1.0, random_state=42))
])

text_clf.fit(X_tr, y_tr)
y_pred = text_clf.predict(X_te)

print('Text Classification Results:')
print(f'Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_te, y_pred, target_names=news.target_names))

In [ ]:
# Top words per category
vec = text_clf.named_steps['tfidf']
clf = text_clf.named_steps['clf']
feature_names = vec.get_feature_names_out()
print('Top 5 words per category:')
for i, cat in enumerate(news.target_names):
    top_idx = clf.coef_[i].argsort()[-5:][::-1]
    print(f'  {cat}: {[feature_names[j] for j in top_idx]}')

## Practical Lab 3 – OCR with EasyOCR (Synthetic Image)

### Objective
Create a synthetic document image and extract text with EasyOCR.

### Explanation
We use Pillow to create a test image with text (avoiding real documents). EasyOCR uses deep learning models trained on thousands of languages.

In [ ]:
# Lab 3: Create synthetic document image
def create_test_document():
    img = Image.new('RGB', (500, 300), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    lines = [
        'AI/ML Training Program',
        'Day 10: NLP and OCR',
        '',
        'Student: Aung Kyaw',
        'Score: 95 / 100',
        'Date: 2026-08-24',
        '',
        'Result: PASS'
    ]
    y_pos = 20
    for line in lines:
        draw.text((30, y_pos), line, fill=(0, 0, 0))
        y_pos += 32
    return img

doc_img = create_test_document()
plt.figure(figsize=(8, 5))
plt.imshow(doc_img); plt.axis('off')
plt.title('Synthetic Test Document'); plt.show()
print('Image size:', doc_img.size)

In [ ]:
# Run EasyOCR
import easyocr
import numpy as np

print('Initializing EasyOCR (downloads model on first run ~100MB)...')
reader = easyocr.Reader(['en'], gpu=False, verbose=False)

# Convert PIL image to numpy array
img_array = np.array(doc_img)

# Extract text
results = reader.readtext(img_array)

print('\nOCR Results:')
print(f'{"Detected Text":<35} {"Confidence":>12}')
print('-' * 50)
for (bbox, text, conf) in results:
    print(f'{text:<35} {conf:>12.4f}')

In [ ]:
# Visualize OCR bounding boxes
import cv2

img_vis = img_array.copy()
for (bbox, text, conf) in results:
    if conf > 0.3:
        pts = np.array(bbox, dtype=np.int32)
        cv2.polylines(img_vis, [pts], True, (0, 255, 0), 2)
        cv2.putText(img_vis, f'{conf:.2f}', tuple(pts[0]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

plt.figure(figsize=(8, 5))
plt.imshow(img_vis)
plt.title('OCR Detections (green boxes, confidence in blue)')
plt.axis('off'); plt.show()

# Reconstruct full text
full_text = '\n'.join([text for _, text, conf in results if conf > 0.3])
print('\nExtracted text:')
print(full_text)

## Practical Lab 4 – PaddleOCR Workflow

### Objective
Understand the PaddleOCR pipeline for scanned PDF processing.

### Explanation
PaddleOCR is one of the most accurate OCR engines. For scanned PDFs, the workflow is:
1. Convert PDF pages to images (PyMuPDF)
2. Preprocess images
3. Run PaddleOCR
4. Reconstruct text with confidence filtering

> **Note:** PaddleOCR requires paddlepaddle backend. The installation below is shown for Google Colab. Run Step 1 to install, then restart the runtime.

In [ ]:
# Lab 4: PaddleOCR Installation (run this, then restart runtime if needed)
# This shows the installation commands without running PaddleOCR to avoid runtime issues
print('PaddleOCR Installation Commands:')
print('  !pip install paddlepaddle -f https://www.paddlepaddle.org.cn/whl/linux/cpu-mkl/avx/stable.html')
print('  !pip install paddleocr pymupdf')
print()
print('PaddleOCR Usage Example:')
code_example = '''
from paddleocr import PaddleOCR
import fitz  # PyMuPDF

# Initialize
ocr = PaddleOCR(use_angle_cls=True, lang='en')

# Convert PDF to images
pdf = fitz.open('document.pdf')
for page_num in range(len(pdf)):
    page = pdf[page_num]
    mat = fitz.Matrix(2.0, 2.0)  # 2x zoom
    pix = page.get_pixmap(matrix=mat)
    img_path = f'page_{page_num}.png'
    pix.save(img_path)
    
    # Run OCR
    result = ocr.ocr(img_path, cls=True)
    for line in result:
        for box, (text, score) in line:
            if score > 0.8:  # confidence filter
                print(f'{text}  (conf={score:.2f})')
'''
print(code_example)

In [ ]:
# Lab 4: Simulate OCR output analysis (without needing PaddleOCR installed)
np.random.seed(42)
simulated_results = [
    {'text': 'Invoice Number: INV-2026-001', 'confidence': 0.98},
    {'text': 'Date: August 24, 2026',         'confidence': 0.95},
    {'text': 'Customer: Ministry of Transport', 'confidence': 0.89},
    {'text': 'Item: AI Training Course',        'confidence': 0.92},
    {'text': 'Amount: 5000 USD',                'confidence': 0.97},
    {'text': 'Signatur3 here',                  'confidence': 0.61},  # OCR error
    {'text': '??corrupted_text??',               'confidence': 0.22},  # bad quality
    {'text': 'Total: 5000 USD',                  'confidence': 0.94},
]

df_ocr = pd.DataFrame(simulated_results)
print('All OCR Results:')
print(df_ocr.to_string(index=False))

HIGH_CONF = 0.85
df_high = df_ocr[df_ocr['confidence'] >= HIGH_CONF]
print(f'\nHigh confidence results (>={HIGH_CONF}):')
print(df_high.to_string(index=False))

## Practical Lab 5 – OCR Confidence Scoring & Post-processing

### Objective
Analyze, filter, and clean OCR output systematically.

### Explanation
OCR is never 100% accurate. Post-processing rules (confidence thresholds, pattern matching, spell checking) improve the final text quality.

In [ ]:
# Lab 5: OCR post-processing pipeline
def filter_by_confidence(results, threshold=0.85):
    return [r for r in results if r['confidence'] >= threshold]

def fix_common_ocr_errors(text):
    replacements = {
        '0': ['O', 'o'] if text.isdigit() else [],  # context-dependent
        'l': ['I', '1'],   # lowercase L vs I vs 1
    }
    text = text.replace('|', 'I')   # pipe → I
    text = text.replace('3', 'e') if 'tur' in text else text  # signature typo
    return text

def extract_fields(results, fields):
    """Extract specific fields from OCR text using patterns."""
    extracted = {}
    for r in results:
        text = r['text']
        for field, pattern in fields.items():
            if field not in extracted:
                import re
                match = re.search(pattern, text)
                if match:
                    extracted[field] = match.group(1)
    return extracted

# Apply pipeline
filtered = filter_by_confidence(simulated_results, 0.85)
fields_to_extract = {
    'invoice_num': r'Invoice Number:\s*(.+)',
    'date':        r'Date:\s*(.+)',
    'amount':      r'Amount:\s*(.+)',
    'total':       r'Total:\s*(.+)',
}
extracted = extract_fields(filtered, fields_to_extract)

print('Post-processing pipeline results:')
print(f'  Total OCR hits:    {len(simulated_results)}')
print(f'  After confidence filter (>0.85): {len(filtered)}')
print(f'\nExtracted structured fields:')
for k, v in extracted.items():
    print(f'  {k}: {v}')

In [ ]:
# Confidence score visualization
df_sim = pd.DataFrame(simulated_results)
plt.figure(figsize=(10, 5))
bars = plt.barh(range(len(df_sim)), df_sim['confidence'],
                color=['green' if c >= 0.85 else 'orange' if c >= 0.6 else 'red'
                        for c in df_sim['confidence']])
plt.yticks(range(len(df_sim)), [t[:40] for t in df_sim['text']], fontsize=8)
plt.axvline(x=0.85, color='blue', linestyle='--', lw=1.5, label='Threshold (0.85)')
plt.xlabel('OCR Confidence Score')
plt.title('OCR Results: Confidence Score Analysis')
plt.legend(); plt.tight_layout(); plt.show()

## Exercises

**Exercise 1:** Add 10 more email samples to the spam classifier from Day 5. Retrain and test on 3 new emails.

In [ ]:
# Exercise 1
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
# Build a larger spam dataset and train

**Exercise 2:** Use 20 Newsgroups with 6 categories instead of 4. Compare accuracy.

In [ ]:
# Exercise 2
from sklearn.datasets import fetch_20newsgroups
# Use categories = ['sci.space','sci.med','comp.graphics','rec.sport.hockey','talk.politics.misc','alt.atheism']

**Exercise 3:** Create a synthetic image with noisy background (add pepper noise with numpy). Compare EasyOCR results before and after denoising.

In [ ]:
# Exercise 3
from PIL import Image, ImageDraw, ImageFilter
import numpy as np
# Create noisy image → run EasyOCR → denoise → run EasyOCR again

## Challenge Task – Receipt OCR Pipeline

Build an end-to-end receipt text extraction system:
1. Create a synthetic receipt image with PIL (shop name, items, prices, total)
2. Run EasyOCR
3. Filter results by confidence (>0.8)
4. Parse total amount using regex
5. Build a structured DataFrame of items and prices

In [ ]:
# Challenge Task: Receipt OCR
def create_receipt():
    img = Image.new('RGB', (400, 500), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    receipt_text = [
        '         HAPPY MART',
        '     123 Main Street, Yangon',
        '-'*40,
        '',
        'Python Book          15.00 USD',
        'NumPy Notebook       12.50 USD',
        'Coffee               3.50 USD',
        'USB Cable            8.99 USD',
        '',
        '-'*40,
        'Subtotal:           39.99 USD',
        'Tax (10%):           4.00 USD',
        '='*40,
        'TOTAL:              43.99 USD',
        '',
        '  Thank you for your purchase!',
        '   Receipt #: RCP-2026-08-001',
    ]
    y_pos = 15
    for line in receipt_text:
        draw.text((20, y_pos), line, fill=(0,0,0))
        y_pos += 28
    return img

receipt = create_receipt()
plt.figure(figsize=(6, 8))
plt.imshow(receipt); plt.axis('off')
plt.title('Synthetic Receipt'); plt.show()
print('Receipt image created. Now run EasyOCR...')
print('(Requires easyocr to be installed and initialized)')

# Simulate structured extraction
receipt_items = [
    {'item': 'Python Book',   'price': 15.00},
    {'item': 'NumPy Notebook','price': 12.50},
    {'item': 'Coffee',        'price': 3.50},
    {'item': 'USB Cable',     'price': 8.99},
]
df_receipt = pd.DataFrame(receipt_items)
df_receipt.loc[len(df_receipt)] = {'item': 'TOTAL', 'price': df_receipt['price'].sum() * 1.1}
print('\nStructured receipt:')
print(df_receipt.to_string(index=False))

## Real-World AI/ML Application

| NLP/OCR Application | Technology |
|---|---|
| Document AI (invoices, forms) | PaddleOCR + NLP extraction |
| Email spam filter | TF-IDF + Logistic Regression |
| News categorization | TF-IDF + Naive Bayes / BERT |
| Sentiment analysis | NLP + LSTM / Transformers |
| Legal document search | TF-IDF + ElasticSearch |
| Passport/ID reading | OCR + template matching |

## Summary

| Concept | Key Point |
|---|---|
| Tokenization | Split text into words/subwords |
| Stopwords | Remove common words that add no meaning |
| TF-IDF | Weight words by frequency × rarity |
| 20 Newsgroups | Multi-class text classification benchmark |
| EasyOCR | Deep learning OCR, 80+ languages |
| PaddleOCR | Best-in-class OCR engine (PDF-aware) |
| Confidence threshold | Filter low-quality OCR results |
| Post-processing | Regex + spell check → structured output |

**→ Day 11: Integrated AI Practical Projects**